# Prepare the datasets

Run this notebook from `evaluation_data/ovarian_cancer/` to acquire GEO metadata
and raw CEL files, apply RMA, and collapse probes to GenBank accessions.
Existing prepared inputs can be used directly with `02_prepare_correction_inputs.ipynb`.


# Harmonize metafiles

In [ ]:
library(tidyverse)
library(GEOquery)

# Selected studies used by every loop below.
datasets <- c("GSE6008", "GSE26712", "GSE40595", "GSE69428", "GSE38666", "GSE10971")

# Metadata acquisition, raw-data processing and probe collapse run only for studies
# without a saved collapsed matrix. Set force_rebuild <- TRUE to recreate all studies.
force_rebuild <- FALSE
studies_to_process <- if (force_rebuild) datasets else
    datasets[!file.exists(file.path("before", datasets, "intermediate", "expr_GBacc_MaxVar.tsv"))]
cat("Studies to process:", paste(studies_to_process, collapse = ", "), "\n")


## Load metadata

i - to check if some dataset contains the data from several sources/platforms

In [ ]:
options(timeout = max(300, getOption("timeout")))
options(download.file.method.GEOquery = "wget")

metadata_files <- list()
samples_IDs <- NULL

for(dataset in studies_to_process){

    dir.create(file.path("before", dataset, "intermediate"), recursive = TRUE, showWarnings = FALSE)

    gset <- getGEO(dataset, GSEMatrix=TRUE, getGPL=FALSE)
    metadata_files[[dataset]] <- list()

    n_platforms <- length(gset)
    for(i in c(1:n_platforms)){
        metadata <- pData(gset[[i]]) %>% as.data.frame()
        write.table(metadata %>% as.data.frame() %>% rownames_to_column("Sample_geo_ID"), 
            file = paste0("before/", dataset, "/intermediate/", i, "_metadata.tsv"), sep = "\t", row.names = F,quote = FALSE)

        metadata_files[[dataset]][[i]] <- paste0("before/", dataset, "/intermediate/", i, "_metadata.tsv")
        if(is.null(samples_IDs)){
            samples_IDs <- metadata %>% rownames()
        }else{
            # add new samples, but without duplicates
            samples_IDs <- c(samples_IDs, metadata %>% rownames()) %>% unique()
        }
    }
}

## Read and preprocess metadata

In [ ]:
prepare_GSE6008 <- function(metadata){
    metadata <- metadata %>%
        # remove all columns with mutation in colname
        select(-c(grep("mutation", colnames(metadata), value = T))) %>%
        select(-c(grep("characteristics_ch1", colnames(metadata), value = T))) %>%
        select(-c("CEL_file_name:ch1", "P53_immunohistochemistry:ch1", "B-canenin_nuclear_accumulation:ch1", "PTEN_immunohistochemistry:ch1")) %>%
        select(-c("title")) %>%
        rename("tissue" = "source_name_ch1", "Grade" = "grade:ch1", "Stage" = "stage:ch1", "Tumor_Type" = "Tumor_Type:ch1") %>%
        mutate(
            Status = ifelse(grepl("Normal", tissue), "normal", "ovarian tumour"),
            Grade2 = ifelse(Grade == 1, "Low", ifelse(Grade == 2 | Grade == 3 | Grade == "2-3", "High", NA)),
            Stage = ifelse(Stage == "N/A", NA, toupper(gsub(" ", "", Stage))),
            Grade = ifelse(Grade == "N/A", NA, Grade),
            Tumor_Type = ifelse(Tumor_Type == "N/A", NA, Tumor_Type),
        ) %>%
        mutate(
            HistSubtypes = ifelse(grepl("Serous", Tumor_Type) & Grade2 == "High", "high-grade serous carcinoma",
                ifelse(grepl("Serous", Tumor_Type) & Grade2 == "Low", "low-grade serous carcinoma",
                        ifelse(grepl("Endometrioid", Tumor_Type), "endometrioid carcinoma",
                            ifelse(grepl("Clear_Cell", Tumor_Type), "clear cell carcinoma",
                                ifelse(grepl("Mucinous", Tumor_Type), "mucinous carcinoma", NA)
                            )
                        )
                    )
                 ),
            ClinicopathologicSubtypes = ifelse(HistSubtypes == "high-grade serous carcinoma", "Type II", "Type I"),
            Stage2 = ifelse(Stage %in% c("3A","3C","3B","3", "3D"), "III", 
                ifelse(Stage %in% c("1A","1C", "1A", "1"), "I", 
                    ifelse(Stage %in% c("4"), "IV", 
                        ifelse(Stage %in% c("2B", "2C", "2A", "2"), "II", NA)))),
            Stage3 = NA
        ) %>%
        mutate(HistSubtypes = ifelse(is.na(HistSubtypes), ifelse(Tumor_Type == "Serous", "serous carcinoma", NA), HistSubtypes)) %>%
        select(-c("Tumor_Type", "description"))
    metadata <- metadata %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
    return(metadata)
}


prepare_GSE26712 <- function(metadata){
    
    metadata <- metadata %>%
        mutate(title = sub(" .*", "", title)) %>%
        rename("tissue" = "source_name_ch1", "Tissue" = "tissue:ch1") %>%
        select(-c("title", "characteristics_ch1", "characteristics_ch1.1", "characteristics_ch1.2", "characteristics_ch1.3",
            "surgery outcome:ch1", "survival years:ch1"))  %>%
        mutate(
            Stage = ifelse(grepl("Late-stage", Tissue), "Late(IIIB–IV)", NA),
            Grade2 = ifelse(grepl("high-grade", Tissue), "High", NA),
            SubtypeRep = ifelse(grepl("high-grade", Tissue), "Ser/PapSer", NA),
            Status = ifelse(grepl("Normal", Tissue), "normal", "ovarian tumour"),
            HistSubtypes = ifelse(SubtypeRep == "Ser/PapSer", "high-grade serous carcinoma", NA)) %>%
        mutate(
            ClinicopathologicSubtypes = ifelse(HistSubtypes == "high-grade serous carcinoma", "Type II", "Type I"),
            Stage2 = Stage,
            Stage3 = NA) %>%
        select(-c("treatment_protocol_ch1", "description", "Tissue"))
    metadata <- metadata %>% 
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
        
    return(metadata)
}

prepare_GSE40595 <- function(metadata){
    metadata <- metadata %>%
        filter(!grepl("stroma", title)) %>%
        mutate(
            Status = ifelse(grepl("Microdissected ovarian surface epthelium", source_name_ch1), "normal", "ovarian tumour"),
            tissue = "Ovary",
            HistSubtypes = ifelse(grepl("tumor", source_name_ch1), "high-grade serous carcinoma", NA),
            Grade2 = ifelse(grepl("tumor", source_name_ch1), "High", NA),
            ClinicopathologicSubtypes = ifelse(grepl("tumor", source_name_ch1), "Type II", NA),
            Stage = NA,
            Stage2 = NA,
            Stage3 = NA
        ) %>%
        select(-c("characteristics_ch1", "title", "tissue:ch1", "description", "source_name_ch1"))
    metadata <- metadata %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
    return(metadata)
}


prepare_GSE69428 <- function(metadata){
    metadata <- metadata %>%
        filter(!grepl("FTSC", title)) %>%
        mutate(
            Grade2 = ifelse(grepl("HGSOC", title), "High", NA),
            ClinicopathologicSubtypes = ifelse(grepl("HGSOC", title), "Type II", NA),
            HistSubtypes = ifelse(grepl("HGSOC", title), "high-grade serous carcinoma", NA),
            Status = ifelse(grepl("normal", title), "normal", "ovarian tumour"),
            tissue = ifelse(grepl("HGSOC", title), "serous ovarian cancer", "oviduct"),
            Stage = NA,
            Stage2 = NA,
            Stage3 = NA
        ) %>%
        select(-c("treatment_protocol_ch1", "characteristics_ch1.1", "characteristics_ch1", 
            "sample type:ch1", "source_name_ch1", "description", "title", "tissue:ch1"))
    metadata <- metadata %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
    return(metadata)
}


prepare_GSE38666 <- function(metadata){
    metadata <- metadata %>%
        filter(!grepl("Stroma", title)) %>%
        select(-c(grep("characteristics_ch1", colnames(metadata), value = T))) %>%
        rename("Stage" = "Stage:ch1", "tissue" = "tissue:ch1", "Grade" = "grade:ch1") %>%
        mutate(
            Status = ifelse(grepl("normal", title), "normal", "ovarian tumour"),
            Stage =  toupper(gsub(" ", "", Stage)),
            Grade2 = ifelse(Grade == 1, "Low", ifelse(Grade == 2 | Grade == 3, "High", NA)),
            HistSubtypes = ifelse(Status == "ovarian tumour" & Grade2 == "High", "high-grade serous carcinoma", NA),
            ClinicopathologicSubtypes = ifelse(HistSubtypes == "high-grade serous carcinoma", "Type II", "Type I"),
            Stage2 = ifelse(Stage %in% c("IIIA","IIIC","IIIB","III", "IIIB/C"), "III", 
                ifelse(Stage %in% c("IB","IC", "IA"), "I", 
                    ifelse(Stage %in% c("III/IV", "IIIC/IV", "IV"), "IV", 
                        ifelse(Stage %in% c("IIB", "IIC", "IIA", "II"), "II", NA)))),
            Stage3 = NA
        ) %>%
        select(-c("source_name_ch1", "description", "cell type:ch1", "title"))
    
    metadata <- metadata %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
    return(metadata)
}


prepare_GSE10971 <- function(metadata){
    # Pre-specified selection (fixed 21 October 2026 before any processing):
    # controls are the 12 LCM fallopian tube epithelium arrays in the FTEn group;
    # the 12 BRCA1/2-carrier arrays (FTEb) are excluded. FTEn BRCA status remains NA. HGSC are
    # one array per patient (11 patients); for patients 2 and 3, who have a tubal and an
    # ovarian array each, the ovary-site array is kept. Tumour site is recorded per array.
    selected_controls <- paste0("GSM2777", 76:87)
    selected_hgsc <- c("GSM277800", paste0("GSM2778", sprintf("%02d", 3:12)))
    characteristics <- metadata %>%
        select(Sample_geo_ID, matches("^characteristics_ch1")) %>%
        pivot_longer(-Sample_geo_ID, values_to = "entry") %>%
        separate(entry, c("key", "value"), sep = ": ", extra = "merge") %>%
        select(-name) %>%
        pivot_wider(names_from = key, values_from = value)
    metadata <- metadata %>%
        left_join(characteristics, by = "Sample_geo_ID") %>%
        filter(Sample_geo_ID %in% c(selected_controls, selected_hgsc)) %>%
        mutate(
            Status = ifelse(grepl("carcinoma", source_name_ch1), "ovarian tumour", "normal"),
            HistSubtypes = ifelse(Status == "ovarian tumour", "high-grade serous carcinoma", NA),
            Grade2 = ifelse(Status == "ovarian tumour", "High", NA),
            ClinicopathologicSubtypes = ifelse(Status == "ovarian tumour", "Type II", NA),
            tumour_site = ifelse(Status == "ovarian tumour",
                ifelse(grepl("of ovary", description), "ovary", "fallopian tube"), NA),
            tissue = ifelse(Status == "ovarian tumour",
                paste("High-grade serous carcinoma,", tumour_site), "Fallopian tube epithelium"),
            donor_id = paste0(ifelse(Status == "ovarian tumour", "HGSC_", "FTEn_"),
                sub(".*patient ([0-9]+).*", "\\1", title)),
            age = as.numeric(Age),
            Stage = NA, Stage2 = NA, Stage3 = NA
        ) %>%
        rename(brca_status = `BRCA1/2 mutation`, cycle_phase = Cycle, family_history = `Family History`) %>%
        select(-c("title", "source_name_ch1", "description", "Age"), -matches("^characteristics_ch1"),
               -matches(":ch1$"))  # GEO characteristic columns duplicate the parsed fields
    stopifnot(nrow(metadata) == 23,
              setequal(metadata$Sample_geo_ID, c(selected_controls, selected_hgsc)),
              sum(metadata$Status == "ovarian tumour") == 11,
              !anyDuplicated(metadata$donor_id))
    metadata <- metadata %>%
        select(Sample_geo_ID, Status, HistSubtypes, Stage, Grade2, ClinicopathologicSubtypes, everything())
    return(metadata)
}


In [ ]:
metadata_list <- list()

for(dataset in studies_to_process){
    cat("Dataset: ", dataset, "\n")
    for(i in c(1:length(metadata_files[[dataset]]))){
        cat("\t\t", i, "\n")
        metadata_file <- metadata_files[[dataset]][[i]]
        metadata <- read_tsv(metadata_file, show_col_types = FALSE)
        # check if metadata$type is RNA
        if("type" %in% colnames(metadata)){
            if(!("RNA" %in% metadata$type)){
                cat("Not RNA dataset\n")
        }}

        metadata <- metadata %>%
            select(-matches("^(geo_accession|status|submission_date|last_update_date|type|channel_count|organism_ch1)")) %>%
            select(-matches("^(molecule_ch1|taxid_ch1|data_row_count|relation|contact_)")) %>%
            select(-matches("^(extract_protocol_ch1|label_ch1|label_protocol_ch1|scan_protocol|hyb_protocol|data_processing)")) %>%
            select(-matches("^(growth_protocol_ch1|data_processing.1|patient id:ch1|age:ch1|supplementary_file.1|gender:ch1)"))
        
        if (dataset == "GSE6008") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE6008(metadata) }
        if (dataset == "GSE26712") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE26712(metadata) }
        if (dataset == "GSE40595") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE40595(metadata) }
        if (dataset == "GSE69428") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE69428(metadata) }
        if (dataset == "GSE38666") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE38666(metadata) }
        if (dataset == "GSE10971") { metadata_list[[paste0(dataset, "_", i)]] <- prepare_GSE10971(metadata) }
    }
}

## Save the results

In [ ]:
library(readr)

for(dataset in names(metadata_list)){
    dataset_path <- gsub("_\\d$", "", dataset)
    # get i from dataset
    i <- gsub(".*_", "", dataset)
    write_tsv(metadata_list[[dataset]], paste0("before/", dataset_path, "/intermediate/", i, "_metadata_short.tsv"))
}


In [ ]:
# Summarize the saved short metadata of every study in the vector, not only the
# studies processed in this run.
saved_metadata <- setNames(
    lapply(datasets, function(d) read_tsv(file.path("before", d, "intermediate", "1_metadata_short.tsv"),
                                          show_col_types = FALSE)),
    paste0(datasets, "_1"))

# Function to process each dataset
process_dataset <- function(dataset) {
    # Read the metadata
    metadata <- saved_metadata[[dataset]]
    # Get unique values for each column and format as a single string if multiple
    summarized <- metadata %>%
        summarise(across(everything(), ~paste(unique(.x), collapse = "; "))) %>%
        pivot_longer(cols = everything(), names_to = "column_name", values_to = "unique_values") %>%
        pivot_wider(names_from = column_name, values_from = unique_values)
    
    # Add dataset identifier
    summarized <- summarized %>% mutate(dataset = dataset)
    summarized$dataset <- as.character(dataset)
    # add column with total number of samples per dataset
    summarized$number_of_samples <- nrow(metadata)
    return(summarized)
}

# Process all datasets
processed_datasets <- map(names(saved_metadata), process_dataset)

# Combine all processed datasets using full join, handling cases with no common columns
final_df <- reduce(processed_datasets, ~full_join(.x, .y, by = intersect(names(.x), names(.y)))) %>%
    as.data.frame()

# Print or view the final structured dataframe
# print(final_df)
final_df %>% write_tsv("preprocessing_info/metadata_summary.tsv", col_names = TRUE)

# Load the data

In [ ]:
library(tidyverse)

library(affy)
library(GEOquery)

In [ ]:
# Preparation (in case of errors)
# library(makecdfenv)

# make.cdf.package(
#     "GPL15048_HuRSTA_2a520709.CDF.gz",
#     packagename = 'hursta2a520709cdf',
#     # package.path = "./",
#     species = "Homo_sapiens",
#     cdf.path = "before/GSE66957_RAW/",
#     compress = TRUE)

# system("R CMD INSTALL hursta2a520709cdf", intern = TRUE)

In [ ]:
options(timeout = max(300, getOption("timeout")))
options(download.file.method.GEOquery = "wget")

for(dataset in studies_to_process){                   
    
    cat('Processing for dataset:', dataset)
    
    if (file.exists(paste0("before/", dataset, "/", dataset,"_RAW.tar"))) {
        untar(
            paste0("before/", dataset,"/", dataset, "_RAW.tar"), 
            exdir = paste0("before/", dataset, "_RAW")
        )
    } else {
        getGEOSuppFiles(dataset, baseDir="before")
        untar(
            paste0("before/", dataset,"/", dataset, "_RAW.tar"), 
            exdir = paste0("before/", dataset, "_RAW")
        )
    }

    # Save the metadata
    # gset <- getGEO(dataset, GSEMatrix=TRUE, getGPL=FALSE)
    gset = c(1)
    
    for(i in c(1:length(gset))){
        metadata <- read.table(paste0("before/", dataset, "/intermediate/", i, "_metadata_short.tsv"), header = TRUE, sep = "\t")
        platform_files <- sub(".*/", "", metadata$supplementary_file)
        
        # if any file contains "txt.gz" ending, remove them from the list
        if(any(grepl("txt.gz", platform_files))){
            platform_files <- platform_files[!grepl("txt.gz", platform_files)]
            cat("Removed txt.gz files from the list. CHECK THE DATASET!")
        }
        
        platform_files_list <- paste0("before/", dataset, "_RAW/", platform_files)

        filenames <- as.character(unlist(platform_files_list))
        rawData <- ReadAffy(
            filenames = unlist(filenames),
            # celfile.path = paste0("before/", dataset, "_RAW"), 
            verbose = FALSE)

        # Prepare sample names
        sample_names <- sampleNames(rawData)
        sample_names <- sub("_.*", "", sample_names)
        sample_names <- sub(".CEL.gz$", "", sample_names)

        # Normalize the data
        normalizedData <- rma(rawData)
        expr_data <- exprs(normalizedData)
    
        # Save the normalized data
        rownames(expr_data) <- featureNames(normalizedData)
        colnames(expr_data) <- sample_names
        write.table(expr_data %>% as.data.frame() %>% rownames_to_column("row_ID"), 
            file = paste0("before/", dataset, "/intermediate/", i, "_expr.tsv"), sep = "\t", row.names = F, quote = FALSE)
        
    }
}

# remove all *_RAW folders from before
system("rm -r before/*_RAW")

## Preprocess the data

In [ ]:
library(tidyverse)
library(WGCNA)

In [ ]:
platform_ids_list <- c()

for(dataset in datasets){
    if(dataset == "GSE131978" | dataset == "GSE68928"){ 
        subsets_len <- c(1, 2)
    } else {subsets_len <- c(1) }

    for(i in subsets_len){
        #read the expression data and metadata file
        metadata <- read.table(paste0("before/", dataset, "/intermediate/", i, "_metadata_short.tsv"), header = TRUE, sep = "\t")
        platform_name <- metadata$platform_id %>% unique()
        platform_ids_list <- c(platform_ids_list, platform_name) %>% unique()
    }
}

In [ ]:
platform_ids_list

In [ ]:
dir.create("preprocessing_info/libraries", recursive = TRUE, showWarnings = FALSE)

geo_platform_table_urls <- c(
    GPL96 = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL96&targ=self&form=text&view=full",
    GPL570 = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL570&targ=self&form=text&view=full"
)

required_geo_annotation_columns <- c("ID", "GB_ACC", "Gene Symbol", "ENTREZ_GENE_ID")

is_geo_annotation_table <- function(path) {
    if(!file.exists(path) || file.info(path)$size == 0) return(FALSE)
    header <- readLines(path, n = 1, warn = FALSE)
    if(length(header) == 0) return(FALSE)
    all(required_geo_annotation_columns %in% strsplit(header, "\t", fixed = TRUE)[[1]])
}

extract_geo_annotation_table <- function(input_path, output_path) {
    input_con <- if(grepl("\\.gz$", input_path)) gzfile(input_path, "rt") else file(input_path, "rt")
    on.exit(close(input_con), add = TRUE)
    lines <- readLines(input_con, warn = FALSE)

    table_begin <- grep("^!.*table_begin", lines, ignore.case = TRUE)
    if(length(table_begin) > 0) {
        start <- table_begin[1] + 1
        table_end <- grep("^!.*table_end", lines, ignore.case = TRUE)
        table_end <- table_end[table_end > start]
        end <- if(length(table_end) > 0) table_end[1] - 1 else length(lines)
    } else {
        start <- grep("^ID\t", lines)[1]
        end <- length(lines)
    }

    if(is.na(start) || start > end) {
        stop("Could not find the GEO annotation table in ", input_path)
    }

    table_lines <- lines[start:end]
    table_lines <- table_lines[nzchar(table_lines)]
    header_columns <- strsplit(table_lines[1], "\t", fixed = TRUE)[[1]]
    missing_columns <- setdiff(required_geo_annotation_columns, header_columns)
    if(length(missing_columns) > 0) {
        stop("GEO annotation table is missing columns: ", paste(missing_columns, collapse = ", "))
    }

    writeLines(table_lines, output_path, useBytes = TRUE)
}

download_geo_platform_annotation <- function(platform_id) {
    output_path <- file.path("preprocessing_info", "libraries", paste0(platform_id, ".txt"))
    if(is_geo_annotation_table(output_path)) {
        cat("Annotation file already exists:", output_path, "\n")
        return(invisible(output_path))
    }

    if(file.exists(output_path) && file.info(output_path)$size > 0) {
        tryCatch({
            extract_geo_annotation_table(output_path, output_path)
            cat("Extracted annotation table from existing file:", output_path, "\n")
            return(invisible(output_path))
        }, error = function(e) {
            cat("Existing annotation file is not directly usable; re-downloading ", platform_id, ".\n", sep = "")
        })
    }

    if(!platform_id %in% names(geo_platform_table_urls)) {
        stop("Missing GEO annotation URL for platform: ", platform_id)
    }

    download_path <- paste0(output_path, ".download")
    download.file(geo_platform_table_urls[[platform_id]], download_path, mode = "wb")
    extract_geo_annotation_table(download_path, output_path)
    unlink(download_path)
    cat("Downloaded annotation file:", output_path, "\n")
    invisible(output_path)
}

invisible(lapply(platform_ids_list, download_geo_platform_annotation))

In [ ]:
libraries_list <- list()

parse_triple <- function(x, delimiter = " /// ") {
    parts <- strsplit(x, delimiter)[[1]]
    # parts <- sort(parts)
    return(parts[1])
}


for (lib_name in platform_ids_list){
    cat("Processing", lib_name, "\n")
    lib_df <- read_tsv(paste0("preprocessing_info/libraries/", lib_name, ".txt"), comment="#", show_col_types = FALSE)
    
    if(lib_name %in% c("GPL570", "GPL96")){
        lib_df <- lib_df %>% 
                  select(ID, GB_ACC, `Gene Symbol`, ENTREZ_GENE_ID) %>%
                  rename(GeneSymbol = `Gene Symbol`) %>%
                  mutate(GeneSymbol = sapply(GeneSymbol, parse_triple, delimiter = " /// "),
                         ENTREZ_GENE_ID = sapply(ENTREZ_GENE_ID, parse_triple, delimiter = " /// "))
    }

    lib_df <- lib_df[!duplicated(lib_df$ID),]
    libraries_list[[lib_name]] <- lib_df
}

In [ ]:
for(lib_name in names(libraries_list)){
    #print len of unqiue values of each column
    cat(lib_name, "\n")
    print(libraries_list[[lib_name]] %>% summarise(across(everything(), ~length(unique(.x)))))
}

In [ ]:
length(intersect(unique(libraries_list[["GPL570"]]$GB_ACC), unique(libraries_list[["GPL96"]]$GB_ACC)))
length(intersect(unique(libraries_list[["GPL570"]]$GeneSymbol), unique(libraries_list[["GPL96"]]$GeneSymbol)))
length(intersect(unique(libraries_list[["GPL570"]]$ENTREZ_GENE_ID), unique(libraries_list[["GPL96"]]$ENTREZ_GENE_ID)))

In [ ]:
platfrom_list <- list()

for(dataset in studies_to_process){
    
    cat('\n\nProcessing for dataset:', dataset, "\n")
    if(dataset == "GSE131978" | dataset == "GSE68928"){ 
        subsets_len <- c(1, 2)
    } else {subsets_len <- c(1) }

    for(i in subsets_len){
        #read the expression data and metadata file
        metadata <- read.table(paste0("before/", dataset, "/intermediate/", i, "_metadata_short.tsv"), header = TRUE, sep = "\t")
        expr_data <- read.table(paste0("before/", dataset, "/intermediate/", i, "_expr.tsv"), header = TRUE, sep = "\t")
        cat('Loaded the data, shape of metadata:', dim(metadata), 'shape of expr_data:', dim(expr_data), '\n')

        platform_name <- metadata$platform_id %>% unique()

        # MaxVar# Collapse to GB_ACC
        gene_names <- libraries_list[[platform_name]]
        gene_names <- gene_names[!is.na(gene_names$GB_ACC),]
        expr_data_2 <- expr_data %>% filter(row_ID %in% gene_names$ID) %>% column_to_rownames("row_ID")
        cat('After filtering, shape of expr_data:', dim(expr_data_2), '\n')

        expr_data_ENTREZ <- WGCNA::collapseRows(
            expr_data_2, 
            rowGroup = gene_names$GB_ACC, 
            rowID = gene_names$ID,
            method="maxRowVariance", connectivityBasedCollapsing=FALSE,
            methodFunction=NULL, connectivityPower=1,
            selectFewestMissing=TRUE, thresholdCombine=NA)$datETcollapsed
        cat('After collapsing rows (Entrez IDs), shape of expr_data:', dim(expr_data_ENTREZ), '\n')
        # save to file
        write.table(expr_data_ENTREZ %>% as.data.frame() %>% rownames_to_column("row_ID"), 
            file = paste0("before/", dataset, "/intermediate/expr_GBacc_MaxVar.tsv"), sep = "\t", row.names = F, quote = FALSE)

}}

In [ ]:
GB_ACC_id_lists <- list()

for(dataset in datasets){                   
    
    cat('\nProcessing for dataset:', dataset, "\n")
    if(dataset == "GSE131978" | dataset == "GSE68928"){ 
        subsets_len <- c(1, 2)
    } else {subsets_len <- c(1) }

    for(i in subsets_len){
        # read data
        expr_data <- read.table(paste0("before/", dataset, "/intermediate/expr_GBacc_MaxVar.tsv"), header = TRUE, sep = "\t")
        # get the  names
        GB_ACC_id_lists[[paste0(dataset, "_", i)]] <- expr_data$row_ID
    }
} 

In [ ]:
library(jsonlite)

write_json(GB_ACC_id_lists, "preprocessing_info/GBacc_IDs.json")